# Normalization of data

In [2]:
import pandas as pd

# Load CSV file as a DataFrame
normalization = pd.read_csv("data/Normalization.csv", delimiter=";")
process_full = pd.read_csv("data/Process.csv", delimiter=";")

# TODO: Select design/target variables
process_raw = process_full.iloc[:, [0,1,5,13,14,17,19,23,26,28,32]]

# Merge normalization factors into process data on product code
process = process_raw.merge(
    normalization[['Product code', 'Normalisation factor']],
    how='left',
    left_on='code',
    right_on='Product code'
)

# Normalize batch-dependent variables
batch_dependent_vars = ['total_waste', 'tbl_fill_mean', 'cyl_height_mean',
                        'Startup_tbl_fill_maxDifference', 'Startup_tbl_fill_mean']
for var in batch_dependent_vars:
    process[var] = process[var] / process['Normalisation factor']

# Drop unnecessary columns
process.drop(columns=['batch', 'code', 'Product code', 'Normalisation factor'], inplace=True)

# Reorder columns
final_columns = [
    'tbl_fill_mean',
    'cyl_height_mean',
    'Startup_tbl_fill_mean',
    'Startup_tbl_fill_maxDifference',
    'ejection_mean',
    'main_CompForce mean',
    'main_CompForce_sd',
    'Total impurities',
    'total_waste'
]
process = process[final_columns]
print(process.head())

   tbl_fill_mean  cyl_height_mean  Startup_tbl_fill_mean  \
0       2.221770         0.874777               2.277778   
1       2.208138         0.877196               2.214773   
2       2.212957         0.880418               2.184167   
3       2.212495         0.876886               2.175521   
4       2.216512         0.885978               2.180417   

   Startup_tbl_fill_maxDifference  ejection_mean  main_CompForce mean  \
0                        0.158333     223.319255             4.255404   
1                        0.075000     215.963899             4.251023   
2                        0.050000     212.530393             4.261263   
3                        0.100000     225.938922             4.357605   
4                        0.079167     237.305389             4.249461   

   main_CompForce_sd  Total impurities  total_waste  
0           0.058473              0.33   885.590278  
1           0.056788              0.34   369.791667  
2           0.054522              0.28

# Linear regression for Total Waste

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Select design variables for linear regression
design_vars = ['tbl_fill_mean', 'cyl_height_mean', 'Startup_tbl_fill_mean', 
               'Startup_tbl_fill_maxDifference', 'ejection_mean', 
               'main_CompForce mean', 'main_CompForce_sd']
target_var = 'total_waste'

X_tw = process[design_vars]
y_tw = process[target_var]

# Split data
X_train_tw, X_test_tw, y_train_tw, y_test_tw = train_test_split(
    X_tw, y_tw, test_size=0.2, random_state=42
)

# Perform linear regression
linear_model = LinearRegression()
linear_model.fit(X_train_tw, y_train_tw)
linear_r2 = linear_model.score(X_test_tw, y_test_tw)

# Print results
print(f"Linear Regression R²: {linear_r2:.4f}")

Linear Regression R²: 0.4871


# Nonlinear models for Total Waste

In [ ]:
from utils.nonlinear import *

# Exclude the PCA variable for nonlinear models
design_vars = ['tbl_fill_mean', 'cyl_height_mean', 'Startup_tbl_fill_mean',
               'Startup_tbl_fill_maxDifference', 'ejection_mean',
               'main_CompForce mean', 'main_CompForce_sd']

X_tw = process[design_vars]
y_tw = process[target_var]

# Split data
X_train_tw, X_test_tw, y_train_tw, y_test_tw = train_test_split(
    X_tw, y_tw, test_size=0.2, random_state=42
)

poly_tw = polynomial_regression(X_test_tw, X_test_tw, y_train_tw, y_test_tw)

# Fine-tune Random Forest hyperparameters
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt']
}

rf_tw = random_forest(X_train_tw, X_test_tw, y_train_tw, y_test_tw, rf_params)

# Fine-tune SVR hyperparameters
svr_params = {
    'svr__kernel': ['rbf'],
    'svr__C': [1, 10],
    'svr__gamma': ['scale']
}

svr_tw = svr(X_train_tw, X_test_tw, y_train_tw, y_test_tw, svr_params)

# Fine-tune Elastic Net hyperparameters
en_params = {
    'poly__degree': [1, 2],
    'elastic__alpha': [0.01, 0.1, 1],
    'elastic__l1_ratio': [0.2, 0.5, 0.8]
}

en_tw = elastic_net(X_train_tw, X_test_tw, y_train_tw, y_test_tw, en_params)

# Fine-tune XGBoost hyperparameters
xgb_params = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'colsample_bytree': [0.8, 1.0],
    'subsample': [1.0],
    'reg_alpha': [0],
    'reg_lambda': [1]
}

xgb_tw = xgb(X_train_tw, X_test_tw, y_train_tw, y_test_tw, xgb_params)


Polynomial Regression (deg 2) R²: 0.6669

Random Forest Test Performance:
Best R² Score: 0.6569
Best Parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}

SVR Test Performance:
Best R² Score: 0.2483
Best Parameters: {'svr__C': 10, 'svr__gamma': 'scale', 'svr__kernel': 'rbf'}


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.726e+04, tolerance: 3.561e+03
  model = cd_fast.enet_coordinate_descent(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.805e+05, tolerance: 3.803e+03
  model = cd_fast.enet_coordinate_descent(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iter


Elastic Net Test Performance:
Best R² Score: 0.4353
Best Parameters: {'elastic__alpha': 0.1, 'elastic__l1_ratio': 0.5, 'poly__degree': 2}

XGBoost Test Performance:
Best R² Score: 0.6953
Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 200, 'reg_alpha': 0, 'reg_lambda': 1, 'subsample': 1.0}


# Nonlinear models for Total Impurities